# Chat Memory & Conversation Persistence

This notebook covers:
- **In-memory history** with `InMemoryChatMessageHistory`
- **`RunnableWithMessageHistory`**: wiring history into any LCEL chain
- **`MessagesPlaceholder`**: the slot in the prompt where history is injected
- **SQLite persistence**: `SQLChatMessageHistory` for durable, cross-session memory
- **Proving persistence**: two separate chain instances reading the same DB

**Key concept:** Chat memory is stored *outside* the LLM. The chain retrieves it before each call and appends new messages after.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory, BaseChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

llm = init_chat_model("gpt-4o-mini")

## 1. The Pattern: RunnableWithMessageHistory

```
User message
      │
      ▼
get_session_history(session_id)  ──► retrieves past messages
      │
      ▼
ChatPromptTemplate
   system: ...
   MessagesPlaceholder("history")  ──► inserts past messages here
   human: {input}
      │
      ▼
   LLM
      │
      ▼
  Response  ──► appended back to history store
```

`RunnableWithMessageHistory` wraps any chain and handles the load/save automatically. You only need to provide:
1. A factory function that returns a `BaseChatMessageHistory` given a `session_id`
2. The key names for `input_messages_key` and `history_messages_key`

In [ ]:
# --- In-memory store: keyed by session_id ---
store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# --- Prompt with history slot ---
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Remember what the user tells you."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain = prompt | llm | StrOutputParser()

# --- Wrap with memory ---
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "demo-user-1"}}

# Turn 1
resp1 = chain_with_history.invoke({"input": "My name is Paulo"}, config)
print(f"Turn 1: {resp1}")

# Turn 2 — model must recall the name from history
resp2 = chain_with_history.invoke({"input": "What's my name?"}, config)
print(f"Turn 2: {resp2}")

print(f"\nMessages in session store: {len(store['demo-user-1'].messages)}")

## 2. Multiple Sessions (session isolation)

Each `session_id` gets its own history. Different users don't share context.

In [ ]:
# User 1
user1_config = {"configurable": {"session_id": "user-alice"}}
chain_with_history.invoke({"input": "I prefer dark mode and Python."}, user1_config)

# User 2
user2_config = {"configurable": {"session_id": "user-bob"}}
chain_with_history.invoke({"input": "I like light mode and JavaScript."}, user2_config)

# Each user gets their own answer
alice = chain_with_history.invoke({"input": "What theme do I prefer?"}, user1_config)
bob   = chain_with_history.invoke({"input": "What theme do I prefer?"}, user2_config)

print(f"Alice: {alice}")
print(f"Bob:   {bob}")

## 3. SQLite Persistence — Surviving Restarts

`SQLChatMessageHistory` persists messages to SQLite. Even if the Python process restarts, history is recovered from disk.

> This is simulated below by creating **two separate chain instances** with the same DB path — the second has zero in-memory state and must load everything from SQLite.

In [ ]:
import os, sqlite3
from langchain_community.chat_message_histories import SQLChatMessageHistory
from sqlalchemy import create_engine

DB_PATH = "./chat_history.db"
SESSION_ID = "persistent_user"

# Clean slate for demo
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

engine = create_engine(f"sqlite:///{DB_PATH}")

def build_chain():
    """Build a fresh chain — simulates a new program run."""
    _llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

    def _get_history(sid: str) -> BaseChatMessageHistory:
        return SQLChatMessageHistory(session_id=sid, connection=engine)

    _prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Remember user preferences and facts."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ])

    return RunnableWithMessageHistory(
        _prompt | _llm | StrOutputParser(),
        _get_history,
        input_messages_key="input",
        history_messages_key="history",
    )

cfg = {"configurable": {"session_id": SESSION_ID}}

# === RUN 1: Store preferences ===
print("--- RUN 1: Storing preferences ---")
chain_v1 = build_chain()
for msg in ["My name is Paulo. I prefer dark mode themes.", "I like Python over JavaScript."]:
    print(f"User: {msg}")
    resp = chain_v1.invoke({"input": msg}, cfg)
    print(f"AI:   {resp}\n")

del chain_v1  # destroy — no in-memory state survives

In [ ]:
# === PROOF: Inspect the raw database ===
print("--- DATABASE PROOF ---")
print(f"DB exists: {os.path.exists(DB_PATH)} | Size: {os.path.getsize(DB_PATH)} bytes")

conn = sqlite3.connect(DB_PATH)
rows = conn.execute("SELECT COUNT(*) FROM message_store").fetchone()[0]
print(f"Messages stored in DB: {rows}")
conn.close()

In [ ]:
# === RUN 2: Brand-new chain, same DB — test recall ===
print("--- RUN 2: Fresh chain, testing recall ---")
chain_v2 = build_chain()

recall_questions = ["What's my name?", "What theme do I prefer?", "What language do I prefer?"]
for q in recall_questions:
    resp = chain_v2.invoke({"input": q}, cfg)
    print(f"User: {q}")
    print(f"AI:   {resp}\n")

del chain_v2

# Cleanup
engine.dispose()
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
print("\nKey insight: The second chain had ZERO in-memory history.")
print("Everything was loaded from SQLite — true persistence!")